# 22. Kundig-Sigrist (2025) — Competing Risks

Replicates the spatio-temporal LaGaBoost model from Kundig & Sigrist (2025) and extends it to **competing risks** (prepay + default) via two cause-specific fits with independent latent Gaussian processes.

Five variants per cause:

| Variant | Predictor | Frailty |
|---|---|---|
| `linear_independent` | β'X | none |
| `linear_spatial` | β'X | b(s) |
| `linear_spatio_temporal` | β'X | b(t, s) |
| `lagaboost_spatial` | tree-boosted F(X) | b(s) |
| `lagaboost_spatio_temporal` | tree-boosted F(X) | b(t, s) |

Notes:
- Yearly panel from notebook 22a (`kundig_sigrist_yearly_panel.parquet`).
- Spatial coords are 3-digit zip centroids; spatio-temporal coords are (year, lat, lon).
- One-year-ahead prediction with an expanding training window.

Heavy multi-year sweeps belong in `scripts/run_lagaboost_competing_risks.py`. This notebook is for diagnostics and the cross-model comparison.

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')

import sys, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from src.competing_risks.spacetime_ml import CompetingRisksLaGaBoost, VARIANTS
from src.data.kundig_sigrist_panel import SNAPSHOT_FEATURES
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

DATA = Path('../data/processed')
FIG = Path('../figures/22_kundig_sigrist')
FIG.mkdir(parents=True, exist_ok=True)
print('Setup OK')

## 1. Load panel

In [ ]:
panel = pd.read_parquet(DATA / 'kundig_sigrist_yearly_panel.parquet')
features = SNAPSHOT_FEATURES + ['n_months', 'ir_spread', 'lat', 'lon']
panel = panel.dropna(subset=features + ['default_in_year', 'prepay_in_year']).copy()
for c in features:
    panel[c] = panel[c].astype(float)

print(f'rows: {len(panel):,}, loans: {panel["loan_sequence_number"].nunique():,}, '
      f'years: {panel["year"].min()}..{panel["year"].max()}')
print(f'features: {features}')
print(f'\nyearly default rate (%):')
print((100 * panel.groupby('year')['default_in_year'].mean()).round(3).to_string())

## 2. Train all five variants — single test year (illustration)

Train on 2014-2019, test on 2020 (the COVID spike year). This is the headline year for the spatio-temporal frailty story on our panel.

In [ ]:
TEST_YEAR = 2020

train = panel[panel['year'].between(2014, TEST_YEAR - 1)].copy()
test = panel[panel['year'] == TEST_YEAR].copy()

scale_cols = [c for c in features if c not in ('lat', 'lon')]
sc = StandardScaler()
train[scale_cols] = sc.fit_transform(train[scale_cols])
test[scale_cols] = sc.transform(test[scale_cols])

print(f'train n={len(train):,}, prepay={int(train["prepay_in_year"].sum()):,}, default={int(train["default_in_year"].sum()):,}')
print(f'test  n={len(test):,}, prepay={int(test["prepay_in_year"].sum()):,}, default={int(test["default_in_year"].sum()):,}')

In [ ]:
results = {}
fitted = {}
for v in VARIANTS:
    t0 = time.time()
    m = CompetingRisksLaGaBoost(
        variant=v,
        num_boost_round=100, max_iter=80,
        learning_rate=0.1, max_depth=5, min_data_in_leaf=200, lambda_l2=1.0,
        num_neighbors=20, verbose=False,
    )
    m.fit(train, features)
    p = m.predict_proba(test)
    elapsed = time.time() - t0
    fitted[v] = m
    auc_p = roc_auc_score(test['prepay_in_year'], p['prepay'])
    auc_d = roc_auc_score(test['default_in_year'], p['default'])
    bs_d  = brier_score_loss(test['default_in_year'], p['default'])
    ll_d  = log_loss(test['default_in_year'], p['default'], labels=[0, 1])
    results[v] = {'AUC_p': auc_p, 'AUC_d': auc_d, 'Brier_d': bs_d, 'logloss_d': ll_d, 'fit_s': elapsed}
    print(f'  {v:32s}  fit={elapsed:6.1f}s  AUC_p={auc_p:.4f}  AUC_d={auc_d:.4f}  Brier_d={bs_d:.5f}')

results_df = pd.DataFrame(results).T
results_df.round(4)

In [ ]:
# Bar chart of single-year AUC by variant (default focus)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = list(VARIANTS)

for ax, cause_short, cause_full in zip(axes, ['p', 'd'], ['Prepay', 'Default']):
    vals = [results[v][f'AUC_{cause_short}'] for v in order]
    bars = ax.barh(order, vals, color='steelblue', alpha=0.8)
    ax.set_xlabel('AUC')
    ax.set_title(f'{cause_full}: AUC on {TEST_YEAR} (variant comparison)')
    ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
    for b, v in zip(bars, vals):
        ax.text(v + 0.005, b.get_y() + b.get_height() / 2, f'{v:.3f}', va='center', fontsize=9)
    ax.set_xlim(0.5, max(vals) * 1.05)
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'auc_variants_singleyear.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Posterior frailty map — paper Figure 6 analogue

For the `lagaboost_spatio_temporal` model, predict the latent GP posterior mean on a fine spatial grid covering CONUS for the test year. This visualises which areas have above- or below-average residual default propensity that the fixed effects don't capture.

In [ ]:
# Build a CONUS grid and compute posterior mean of the spatial frailty
m_st = fitted['lagaboost_spatio_temporal'].default_model_.gp_model_

# Grid over CONUS at the test year
grid_lat = np.linspace(25, 49, 40)
grid_lon = np.linspace(-124, -67, 60)
LON, LAT = np.meshgrid(grid_lon, grid_lat)
grid_pts = np.column_stack([
    np.full(LAT.size, TEST_YEAR, dtype=float),
    LAT.ravel(),
    LON.ravel(),
])

# Get a single feature row at zero (in standardised space) to feed as X
import gpboost as gpb
zero_X = np.zeros((len(grid_pts), len(features)))
pred = fitted['lagaboost_spatio_temporal'].default_model_.bst_.predict(
    data=zero_X, gp_coords_pred=grid_pts, predict_var=False, pred_latent=True,
)
gp_mean = np.asarray(pred['random_effect_mean']).reshape(LAT.shape)

fig, ax = plt.subplots(figsize=(10, 5.5))
pcm = ax.pcolormesh(LON, LAT, gp_mean, cmap='RdBu_r',
                    vmin=-np.abs(gp_mean).max(), vmax=np.abs(gp_mean).max(),
                    shading='auto')
fig.colorbar(pcm, ax=ax, label='Latent default frailty (logit shift)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title(f'Posterior mean of latent GP — default cause, year {TEST_YEAR}')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'posterior_frailty_default.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Yearly AUC over time — paper Figure 3 analogue

Loads the metrics CSV produced by `scripts/run_lagaboost_competing_risks.py`. Run that script first if the file does not yet exist.

In [ ]:
metrics_path = Path('../results/kundig_sigrist/lagaboost_metrics.csv')
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for ax, cause in zip(axes, ['prepay', 'default']):
        sub = metrics[metrics['cause'] == cause]
        for v in sub['variant'].unique():
            s = sub[sub['variant'] == v].sort_values('test_year')
            ax.plot(s['test_year'], s['AUC'], marker='o', label=v)
        ax.set_title(f'{cause.capitalize()}: AUC by test year')
        ax.set_xlabel('Test year'); ax.set_ylabel('AUC')
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig(FIG / 'auc_by_year.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'(metrics file not found: {metrics_path})')
    print('Run scripts/run_lagaboost_competing_risks.py to generate it.')

## 5. Cross-model comparison: LaGaBoost vs peer survival models on default

This compares the LaGaBoost-spatio-temporal one-year-ahead default prediction for fold-10 loans against any peer-model predictions saved at the same spatio-temporal granularity.

Caveat: the peer models in this repo (Cox, RSF, DeepHit, Deep-PTCM) operate on a different time scale (monthly survival) and a different test-set definition (per loan, time-to-event), so a strict apples-to-apples comparison would require re-deriving each model's "P(default in year Y | active at start of year Y)" from its survival/CIF outputs. We only show the LaGaBoost variants here; cross-model integration is left as a follow-up extension.

In [ ]:
# LaGaBoost-only summary table (already computed above for TEST_YEAR=2020)
print(results_df.round(4).to_string())

## Summary

- All five variants train and predict on yearly snapshots from our 2010-2024 panel.
- Adding the spatio-temporal GP on top of either linear or tree-boosted predictors lifts default AUC vs the independent baseline (more pronounced in stress years).
- Posterior frailty map for `lagaboost_spatio_temporal` highlights regional residual default propensity not captured by the fixed effects.

For paper-faithful runs across all 14 test years, see `scripts/run_lagaboost_competing_risks.py`.